# AI Translation Matcher – Textual Match then Visual Refinement

This notebook begins by searching for documents similar to an input file using a **text** search (TF-IDF + NearestNeighbors), then applies a **visual** search using MobileNetV2 only to textually similar documents.

- Text search using `output/text_vectors.pkl`
- Visual refinement using `output/image_vectors.pkl`
- Objective: To test the effectiveness of combined text > image matching

### Imports and interactive file selection

In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../../src"))

import pickle
import numpy as np
import pandas as pd
import time
from PIL import Image
from IPython.display import display
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

from model import extract_text
from fetch import download_file_from_space
from image_features_mobilenet import extract_visual_features

import fitz
from docx import Document
import io

import ipywidgets as widgets
from IPython.display import display as ipy_display

test_dir = "../../AI_matcher/tests"

available_files = sorted([
    f for f in os.listdir(test_dir)
    if os.path.isfile(os.path.join(test_dir, f)) and not f.startswith(".DS_Store")
])

dropdown = widgets.Dropdown(
    options=available_files,
    description="File:",
    disabled=False
)

ipy_display(dropdown)

### Parameters

In [ ]:
query_path = os.path.join(test_dir, dropdown.value)
text_index_path = "../../output/text_vectors.pkl"
image_index_path = "../../output/image_vectors.pkl"
textual_threshold = 70.0
top_k_visual = 5

### Text-based similarity search (TF-IDF)

In [ ]:
with open(text_index_path, "rb") as f:
    tfidf_data = pickle.load(f)

text_keys = tfidf_data["keys"]
text_matrix = normalize(tfidf_data["vectors"])
vectorizer = tfidf_data["vectorizer"]

with open(query_path, "rb") as f:
    query_content = f.read()
query_text = extract_text(query_content, query_path)

query_vec = normalize(vectorizer.transform([query_text]))

nn_text = NearestNeighbors(n_neighbors=len(text_keys), metric="cosine")
nn_text.fit(text_matrix)
distances, indices = nn_text.kneighbors(query_vec)

matches_text = []
for j, i in enumerate(indices[0]):
    score_pct = (1 - distances[0][j]) * 100
    if score_pct >= textual_threshold:
        matches_text.append((text_keys[i], score_pct))

df_text_matches = pd.DataFrame(matches_text, columns=["file", "similarity_text_%"])
print(f"Found {len(df_text_matches)} documents with textual similarity ≥ {textual_threshold}%")

### Load visual vector index

In [ ]:
with open(image_index_path, "rb") as f:
    image_data = pickle.load(f)

image_keys = list(image_data.keys())
image_vectors = list(image_data.values())
image_index_map = dict(zip(image_keys, image_vectors))

### Visual refinement of matched documents

In [ ]:
query_visual_vec = None
if query_path.lower().endswith((".jpg", ".jpeg", ".png")):
    query_visual_vec = extract_visual_features(query_content)
else:
    print(f"[INFO] Visual refinement skipped: '{query_path}' is not an image file.")


results_visual = []
if query_visual_vec is not None:
    for file_key in df_text_matches["file"]:
        if file_key in image_index_map:
            vec = image_index_map[file_key]
            dist = np.linalg.norm(query_visual_vec - vec)
            score = round(100 - dist, 2)
            results_visual.append((file_key, score))

df_visual = pd.DataFrame(results_visual, columns=["file", "similarity_visual_%"])
df_merged = df_text_matches.merge(df_visual, on="file", how="left").sort_values(by="similarity_visual_%", ascending=False)

### Display combined results

In [ ]:
print("\nCombined results (text match ≥ 70% + visual score):")
print(df_merged.head(5))

### Preview function for any document type

In [ ]:
from IPython.display import Image as IPyImage

def display_document_preview(file_key: str):
    print(f"\nPreview: {file_key}")
    try:
        content = download_file_from_space(file_key)
        if file_key.lower().endswith((".jpg", ".jpeg", ".png")):
            img = Image.open(io.BytesIO(content))
            display(img)
        elif file_key.lower().endswith(".pdf"):
            doc = fitz.open(stream=content, filetype="pdf")
            pix = doc[0].get_pixmap(dpi=150)
            ipy_img = IPyImage(data=pix.tobytes("png"))
            display(ipy_img)
        elif file_key.lower().endswith(".docx"):
            doc = Document(io.BytesIO(content))
            for rel in doc.part._rels:
                rel_obj = doc.part._rels[rel]
                if "image" in rel_obj.target_ref:
                    img_data = rel_obj.target_part.blob
                    img = Image.open(io.BytesIO(img_data))
                    display(img)
                    break
            else:
                print("[INFO] No image found in the .docx file.")
        else:
            print("[SKIP] Unsupported file type for preview.")
    except Exception as e:
        print(f"[ERROR] Could not display {file_key}: {e}")

### Display visual previews of matched documents

In [ ]:
print("\nVisual previews of top 5 results:")
for file_key in df_merged.head(5)["file"]:
    display_document_preview(file_key)